# Customer Churn Analysis & Retention Strategy

Run cells in order using the project environment. Phase 1 Step 2 audits the raw CSV without changing it. Outputs are aggregate evidence, not retention recommendations. Later phases remain planned.

Business framing: [analysis brief](../outputs/reports/analysis_brief.md). Decisions and audit evidence: [analysis log](../outputs/reports/analysis_log.md).

## Phase 1: Business framing and data audit

### Step 2.1: Input identity and loading contract

Load both a literal string view (preserving blanks and NA-like tokens) and a pandas-inferred view for dtype/null comparison. Neither is cleaned. No customer-level rows are displayed. The timestamp uses Asia/Calcutta; provenance, field units, and the business churn window remain unverified.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import csv
import hashlib
import platform
import numpy as np
import pandas as pd
from IPython.display import display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

project_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / "datasets/TelecomChurnData.csv").is_file()), None)
if project_root is None:
    raise FileNotFoundError("Run from inside the CustomerChurnPrediction repository.")
raw_data_path = project_root / "datasets/TelecomChurnData.csv"
source_hash = hashlib.sha256(raw_data_path.read_bytes()).hexdigest()
load_options = dict(encoding="utf-8", sep=",", header=0)
raw_df = pd.read_csv(raw_data_path, **load_options)
literal_df = pd.read_csv(raw_data_path, dtype="str", keep_default_na=False,
                         na_filter=False, **load_options)
raw_snapshot, literal_snapshot = raw_df.copy(deep=True), literal_df.copy(deep=True)
audit_time = datetime.now(ZoneInfo("Asia/Calcutta")).isoformat()
metadata = pd.Series({"audit_time": audit_time, "input": "datasets/TelecomChurnData.csv",
    "sha256": source_hash, "python": platform.python_version(), "pandas": pd.__version__,
    "numpy": np.__version__, "rows": len(raw_df), "columns": raw_df.shape[1],
    "loading": str(load_options), "inferred_view": "pandas defaults, including default NA parsing",
    "literal_view": "dtype=str, keep_default_na=False, na_filter=False"}, name="value")
display(metadata.to_frame())

,value
audit_time,2026-11-06T21:53:48.721940+05:30
input,datasets/TelecomChurnData.csv
sha256,88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a
python,3.14.6
pandas,3.1.1
numpy,2.5.4
rows,7043
columns,21
loading,"{'encoding': 'utf-8', 'sep': ',', 'header': 0}"
inferred_view,"pandas defaults, including default NA parsing"


### Step 2.2: Schema, customer identifiers, and duplicates

Compare with the dictionary's expected schema, including a direct CSV header check for duplicate names. Repeated-ID counts include all participating rows; exact duplicate counts distinguish excess copies from all participants. A conflicting ID has more than one distinct literal record. These populations may overlap.

In [2]:
reports = project_root / "outputs/reports"
dictionary_path = reports / "data_dictionary.csv"
dictionary_seed = pd.read_csv(dictionary_path, keep_default_na=False)
expected_fields = dictionary_seed["field"].tolist()
with raw_data_path.open(encoding="utf-8", newline="") as source:
    header = next(csv.reader(source))
schema = pd.Series({"missing_fields": sorted(set(expected_fields) - set(header)),
    "unexpected_fields": sorted(set(header) - set(expected_fields)),
    "duplicate_header_names": sorted({f for f in header if header.count(f) > 1})}, name="result")
display(schema.to_frame())
ids = literal_df["customerID"]
valid_id = ids.str.strip().ne("")
repeated_id = valid_id & ids.duplicated(keep=False)
distinct_records = literal_df.loc[valid_id].drop_duplicates()
id_record_counts = distinct_records.groupby("customerID", sort=False).size()
conflict_ids = id_record_counts.index[id_record_counts.gt(1)]
identifier_audit = pd.Series({
    "inferred_null_ids": int(raw_df["customerID"].isna().sum()),
    "empty_ids": int(ids.eq("").sum()),
    "whitespace_only_ids": int((ids.ne("") & ids.str.strip().eq("")).sum()),
    "unique_nonblank_ids": int(ids[valid_id].nunique()),
    "exact_duplicate_excess_rows": int(literal_df.duplicated().sum()),
    "exact_duplicate_participating_rows": int(literal_df.duplicated(keep=False).sum()),
    "repeated_id_rows": int(repeated_id.sum()),
    "repeated_id_groups": int(ids[repeated_id].nunique()),
    "conflicting_id_groups": len(conflict_ids),
    "conflicting_id_rows": int(ids.isin(conflict_ids).sum()),
    "ids_with_outer_whitespace": int(ids.ne(ids.str.strip()).sum()),
    "distinct_ids_collapsing_after_strip_casefold": int(ids[valid_id].nunique() - ids[valid_id].str.strip().str.casefold().nunique()),
}, name="count")
display(identifier_audit.to_frame())

,result
missing_fields,[]
unexpected_fields,[]
duplicate_header_names,[]


,count
inferred_null_ids,0
empty_ids,0
whitespace_only_ids,0
unique_nonblank_ids,7043
exact_duplicate_excess_rows,0
exact_duplicate_participating_rows,0
repeated_id_rows,0
repeated_id_groups,0
conflicting_id_groups,0
conflicting_id_rows,0


### Step 2.3: Field completeness and text consistency

The literal CSV has no typed null representation. `inferred_nulls` counts values interpreted as missing by pandas; literal empty strings, nonempty whitespace-only strings, and nonblank NA-like tokens parsed as null are reported separately. Category frequencies retain original spelling and service states. Normalised views are diagnostic only.

In [3]:
missing_rows = []
for field in header:
    text = literal_df[field]
    empty = text.eq("")
    whitespace = text.ne("") & text.str.strip().eq("")
    missing_rows.append({"field": field, "observed_dtype": str(raw_df[field].dtype),
        "inferred_nulls": int(raw_df[field].isna().sum()), "empty_strings": int(empty.sum()),
        "whitespace_only": int(whitespace.sum()),
        "nonblank_na_tokens": int((raw_df[field].isna() & ~(empty | whitespace)).sum()),
        "outer_whitespace": int(text.ne(text.str.strip()).sum()),
        "distinct_literal_values": int(text.nunique())})
field_profile = pd.DataFrame(missing_rows).set_index("field")
display(field_profile)
numeric_fields = ["tenure", "MonthlyCharges", "TotalCharges"]
category_fields = [f for f in header if f not in ["customerID", *numeric_fields]]
frequency_rows, category_checks = [], []
for field in category_fields:
    text = literal_df[field]
    mapping = pd.DataFrame({"raw": text, "normalised": text.str.strip().str.casefold()}).drop_duplicates()
    collisions = mapping["normalised"].value_counts().gt(1)
    category_checks.append({"field": field, "outer_whitespace_rows": int(text.ne(text.str.strip()).sum()),
        "normalisation_collision_groups": int(collisions.sum())})
    for value, count in text.value_counts(dropna=False).items():
        frequency_rows.append({"field": field, "literal_value": repr(value), "rows": int(count)})
category_audit = pd.DataFrame(category_checks).set_index("field")
category_frequencies = pd.DataFrame(frequency_rows)
display(category_audit)
with pd.option_context("display.max_rows", None):
    display(category_frequencies)
print("SeniorCitizen expected coding {0, 1}; unexpected rows:",
      int((~literal_df["SeniorCitizen"].isin(["0", "1"])).sum()))

SeniorCitizen expected coding {0, 1}; unexpected rows: 0


,observed_dtype,inferred_nulls,empty_strings,whitespace_only,nonblank_na_tokens,outer_whitespace,distinct_literal_values
field,,,,,,,
customerID,str,0,0,0,0,0,7043
gender,str,0,0,0,0,0,2
SeniorCitizen,int64,0,0,0,0,0,2
Partner,str,0,0,0,0,0,2
Dependents,str,0,0,0,0,0,2
tenure,int64,0,0,0,0,0,73
PhoneService,str,0,0,0,0,0,2
MultipleLines,str,0,0,0,0,0,3
InternetService,str,0,0,0,0,0,3


,outer_whitespace_rows,normalisation_collision_groups
field,,
gender,0,0
SeniorCitizen,0,0
Partner,0,0
Dependents,0,0
PhoneService,0,0
MultipleLines,0,0
InternetService,0,0
OnlineSecurity,0,0
OnlineBackup,0,0


,field,literal_value,rows
0,gender,'Male',3555
1,gender,'Female',3488
2,SeniorCitizen,'0',5901
3,SeniorCitizen,'1',1142
4,Partner,'No',3641
5,Partner,'Yes',3402
6,Dependents,'No',4933
7,Dependents,'Yes',2110
8,PhoneService,'Yes',6361
9,PhoneService,'No',682


### Step 2.4: Numeric validity and review flags

Blank values are distinct from nonblank parse failures and non-finite numeric tokens. No source-supported upper bounds are available. Negative values and fractional tenure are flagged; 1.5-IQR fences identify distributional tails for review, not verified errors. No values are imputed, capped, or removed.

In [4]:
numeric_values, numeric_rows = {}, []
for field in numeric_fields:
    text = literal_df[field]
    blank = text.str.strip().eq("")
    values = pd.to_numeric(text.str.strip().mask(blank), errors="coerce").astype("float64")
    token = text.str.strip().str.casefold()
    special = token.isin(["nan", "+nan", "-nan", "inf", "+inf", "-inf", "infinity", "+infinity", "-infinity"])
    finite = pd.Series(np.isfinite(values), index=values.index)
    good = values[finite]
    q1, q3 = good.quantile([0.25, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    numeric_values[field] = values
    numeric_rows.append({"field": field, "blank_rows": int(blank.sum()),
        "nonblank_parse_failures": int((~blank & values.isna() & ~special).sum()),
        "nonfinite_token_rows": int((~blank & (special | (values.notna() & ~finite))).sum()),
        "negative_rows": int((finite & values.lt(0)).sum()),
        "fractional_rows": int((finite & values.mod(1).ne(0)).sum()),
        "finite_count": int(finite.sum()), "minimum": float(good.min()), "maximum": float(good.max()),
        "iqr_lower_fence": float(low), "iqr_upper_fence": float(high),
        "iqr_review_rows": int((finite & (values.lt(low) | values.gt(high))).sum())})
numeric_audit = pd.DataFrame(numeric_rows).set_index("field")
display(numeric_audit)
print("Fractional charges are legitimate numeric values; only fractional tenure is a whole-unit review flag.")
total_blank = literal_df["TotalCharges"].str.strip().eq("")
total_blank_by_tenure = pd.crosstab(numeric_values["tenure"], total_blank).rename(columns={False: "nonblank_TotalCharges", True: "blank_TotalCharges"}).sort_index()
with pd.option_context("display.max_rows", None):
    display(total_blank_by_tenure)
display(pd.Series({"zero_tenure_rows": int(numeric_values["tenure"].eq(0).sum()),
    "blank_total_at_zero_tenure": int((total_blank & numeric_values["tenure"].eq(0)).sum()),
    "blank_total_at_other_tenure": int((total_blank & ~numeric_values["tenure"].eq(0)).sum()),
    "total_below_monthly_review_rows": int((numeric_values["TotalCharges"] < numeric_values["MonthlyCharges"]).sum())}, name="rows").to_frame())
print("TotalCharges below MonthlyCharges is a review flag only: billing timing and units are unverified.")

Fractional charges are legitimate numeric values; only fractional tenure is a whole-unit review flag.
TotalCharges below MonthlyCharges is a review flag only: billing timing and units are unverified.


,blank_rows,nonblank_parse_failures,nonfinite_token_rows,negative_rows,fractional_rows,finite_count,minimum,maximum,iqr_lower_fence,iqr_upper_fence,iqr_review_rows
field,,,,,,,,,,,
tenure,0,0,0,0,0,7043,0.00,72.00,-60.00000,124.00000,0
MonthlyCharges,0,0,0,0,6651,7043,18.25,118.75,-46.02500,171.37500,0
TotalCharges,11,0,0,0,6708,7032,18.80,8684.80,-4688.48125,8884.66875,0


TotalCharges,nonblank_TotalCharges,blank_TotalCharges
tenure,,
0.0,0,11
1.0,613,0
2.0,238,0
3.0,200,0
4.0,176,0
5.0,133,0
6.0,110,0
7.0,131,0
8.0,123,0


,rows
zero_tenure_rows,11
blank_total_at_zero_tenure,11
blank_total_at_other_tenure,0
total_below_monthly_review_rows,0


### Step 2.5: Service-state consistency

Check expected vocabulary and both directions of service dependency. These are schema-based review rules pending source definitions. `No internet service` and `No phone service` are valid states, not missing values.

In [5]:
service_checks = {}
phone, lines = literal_df["PhoneService"], literal_df["MultipleLines"]
service_checks["unexpected_PhoneService"] = ~phone.isin(["Yes", "No"])
service_checks["unexpected_MultipleLines"] = ~lines.isin(["Yes", "No", "No phone service"])
service_checks["phone_dependency_mismatch"] = ((phone.eq("No") & lines.ne("No phone service")) |
    (phone.eq("Yes") & lines.eq("No phone service")))
internet = literal_df["InternetService"]
service_checks["unexpected_InternetService"] = ~internet.isin(["DSL", "Fiber optic", "No"])
for field in ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]:
    service = literal_df[field]
    service_checks[f"unexpected_{field}"] = ~service.isin(["Yes", "No", "No internet service"])
    service_checks[f"internet_dependency_{field}"] = ((internet.eq("No") & service.ne("No internet service")) |
        (internet.isin(["DSL", "Fiber optic"]) & service.eq("No internet service")))
service_audit = pd.Series({name: int(mask.sum()) for name, mask in service_checks.items()}, name="flagged_rows")
display(service_audit.to_frame())
print("Rows with any service flag:", int(pd.DataFrame(service_checks).any(axis=1).sum()))

Rows with any service flag: 0


,flagged_rows
unexpected_PhoneService,0
unexpected_MultipleLines,0
phone_dependency_mismatch,0
unexpected_InternetService,0
unexpected_OnlineSecurity,0
internet_dependency_OnlineSecurity,0
unexpected_OnlineBackup,0
internet_dependency_OnlineBackup,0
unexpected_DeviceProtection,0
internet_dependency_DeviceProtection,0


### Step 2.6: Provisional outcome eligibility and baseline

For this audit only, include a row when its literal ID is nonblank, has no outer whitespace, appears exactly once, and its exact label is `Yes` or `No`. All repeated IDs are held for review; there is no arbitrary duplicate resolution. Report exclusions as a union as well as separate reasons. This is recorded-label prevalence, not a monthly or future-period churn rate. Business grain and outcome meaning remain unverified.

In [6]:
labels = literal_df["Churn"]
display(labels.value_counts(dropna=False).rename_axis("literal_label").to_frame("rows"))
outcome_audit = pd.Series({"empty_labels": int(labels.eq("").sum()),
    "whitespace_only_labels": int((labels.ne("") & labels.str.strip().eq("")).sum()),
    "unexpected_nonblank_labels": int((labels.str.strip().ne("") & ~labels.isin(["Yes", "No"])).sum())}, name="rows")
display(outcome_audit.to_frame())
exclusion_reasons = pd.DataFrame({"blank_id": ~valid_id,
    "id_outer_whitespace": ids.ne(ids.str.strip()), "repeated_id": repeated_id,
    "invalid_label": ~labels.isin(["Yes", "No"])})
eligible = ~exclusion_reasons.any(axis=1)
eligible_count = int(eligible.sum())
churn_count = int((eligible & labels.eq("Yes")).sum())
baseline = pd.Series({"raw_rows": len(literal_df), "eligible_unique_customers": eligible_count,
    "excluded_rows_union": int((~eligible).sum()), "churned_customers": churn_count,
    "retained_customers": int((eligible & labels.eq("No")).sum()),
    "provisional_recorded_churn_pct": 100 * churn_count / eligible_count if eligible_count else None}, name="value")
display(exclusion_reasons.sum().rename("rows").to_frame())
display(baseline.to_frame())
assert literal_df.loc[eligible, "customerID"].is_unique
assert eligible_count + int((~eligible).sum()) == len(literal_df)
assert churn_count + int((eligible & labels.eq("No")).sum()) == eligible_count

,rows
literal_label,
No,5174
Yes,1869


,rows
empty_labels,0
whitespace_only_labels,0
unexpected_nonblank_labels,0


,rows
blank_id,0
id_outer_whitespace,0
repeated_id,0
invalid_label,0


,value
raw_rows,7043.000000
eligible_unique_customers,7043.000000
excluded_rows_union,0.000000
churned_customers,1869.000000
retained_customers,5174.000000
provisional_recorded_churn_pct,26.536987


### Step 2.7: Dictionary export and preservation checks

Export aggregate field metadata only. Descriptions and expected types remain provisional source interpretations; observed structure and counts are verified for this input checksum. No customer IDs or individual records enter the dictionary. Re-execution refreshes this CSV; source data is never written.

In [7]:
dictionary_rows = []
for field in header:
    seed = dictionary_seed.set_index("field").loc[field]
    p = field_profile.loc[field]
    role = "identifier; exclude from model features" if field == "customerID" else ("outcome" if field == "Churn" else "descriptive feature; model timing unverified")
    if field in numeric_fields:
        observed = f"finite range {numeric_audit.loc[field, 'minimum']:g} to {numeric_audit.loc[field, 'maximum']:g}"
        units = "Unverified relationship-length unit" if field == "tenure" else "Unverified currency and billing timing"
        notes = "Numeric diagnostic conversion only; raw unchanged. See notebook numeric checks."
    elif field == "customerID":
        observed = f"{ids[valid_id].nunique()} distinct nonblank identifiers; values withheld"
        units = "Identifier; customer/account/subscription meaning unverified"
        notes = "See notebook identifier integrity checks; observed uniqueness does not verify business grain."
    else:
        observed = "; ".join(f"{value!r}: {count}" for value, count in literal_df[field].value_counts().items())
        units = "Observed 0/1 coding; source definition unverified" if field == "SeniorCitizen" else "Category; source definition unverified"
        notes = "Preserve structural no-service categories." if field in ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"] else "Observed categories are not proof of source-allowed vocabulary."
        if field == "Churn":
            notes = "Yes/No observed coding; event definition and observation window unverified."
    dictionary_rows.append({"field": field, "expected_type": seed["expected_type"],
        "description": seed["description"], "observed_type": p["observed_dtype"],
        "units_or_coding": units, "observed_values_or_range": observed,
        "inferred_null_count": int(p["inferred_nulls"]), "empty_string_count": int(p["empty_strings"]),
        "whitespace_only_count": int(p["whitespace_only"]), "nonblank_na_token_count": int(p["nonblank_na_tokens"]),
        "analytical_role": role, "verification_status": "Observed profile verified; source semantics unverified",
        "notes": notes, "source_sha256": source_hash})
dictionary = pd.DataFrame(dictionary_rows)
assert dictionary["field"].tolist() == header
assert dictionary["field"].is_unique
pd.testing.assert_frame_equal(raw_df, raw_snapshot)
pd.testing.assert_frame_equal(literal_df, literal_snapshot)
assert hashlib.sha256(raw_data_path.read_bytes()).hexdigest() == source_hash
dictionary.to_csv(dictionary_path, index=False, encoding="utf-8", lineterminator="\n")
print("Dictionary written:", dictionary_path.relative_to(project_root))
print("Raw file checksum unchanged:", source_hash)
print("Both in-memory raw views unchanged. No cleaned dataset generated.")

Dictionary written: outputs\reports\data_dictionary.csv
Raw file checksum unchanged: 88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a
Both in-memory raw views unchanged. No cleaned dataset generated.


### Step 2 review

Review the displayed schema, identifiers, missingness, categories, numeric checks, service consistency, and provisional label baseline together with the analysis log. Source provenance, permitted usage, field units, churn timing, and business grain remain open. Step 3 will consolidate the quality report and agree on preparation treatments; no imputations or exclusions have been applied to the source.

## Phase 2: Data cleaning and analytical preparation

**Planned:** Investigate the audit results, document treatments, validate customer uniqueness, and define the churn indicator and analytical bands. Reconcile counts before and after preparation.

Keep `raw_df` unchanged and create a separate analytical table. Save generated tables in `outputs/data/` only after the cleaning rules are agreed and documented. Fit learned model preprocessing on training data in Phase 5.


## Phase 3: Exploratory analysis and SQL validation

**Planned:** Answer the README's business questions using contract, tenure, charges, billing, and service comparisons. Show group sizes, churn volumes, and churn rates together.

Add reproducible queries in `sql/`, reconcile their outputs with pandas, and save reviewed charts in `outputs/charts/`. Record evidence and limitations in the analysis log.


## Phase 4: Business interpretation and retention planning

**Planned:** Prioritise segments by evidence, scale, and operational feasibility. Distinguish observed associations from explanations and proposed actions.

Use the recommendation template in the analysis log to define the target segment, action, owner, success measure, and pilot design. Campaign execution requires additional operational data.


## Phase 5: Baseline predictive support

**Planned:** Verify feature timing, split before fitting preprocessing, and compare a dummy classifier, logistic regression, and a simple segmentation rule.

Choose thresholds using training-side validation. Evaluate held-out precision, recall, and lift at a documented contact capacity. Record limitations and any generated artifacts in `outputs/models/`; add model dependencies when implementing this phase.


## Phase 6: Dashboard, reporting, and portfolio delivery

**Planned:** Build the executive overview, segment analysis, and retention priorities pages described in `dashboard/README.md`.

Reconcile dashboard metrics with Python and SQL, write the executive summary, and update the project README with validated findings and artifact links.
